# Text-only baseline \u2014 Subtask 1A

Fine-tune an Arabic BERT model on the OCR-extracted meme text to classify each meme as `Hateful` / `Not Hateful`.

This notebook is the Jupyter version of [`baselines/train_text.py`](../train_text.py). The script supports both subtasks (`1a`, `1b`); here we focus on `1a` and show how to adapt it for `1b` at the end.

**Prerequisites:** `python data/download_data.py` has been run, and `pip install -r requirements.txt` is done.

In [ ]:
import sys, json, os, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
)

TASK1 = Path.cwd().resolve()
while TASK1.name != 'task1' and TASK1.parent != TASK1:
    TASK1 = TASK1.parent
sys.path.insert(0, str(TASK1 / 'baselines'))

from io_utils import read_jsonl, write_subtask_1a_tsv
from labels import get_task

DATA = TASK1 / 'data'

SUBTASK = '1a'
MODEL_NAME = 'aubmindlab/bert-base-arabertv02'
MAX_LEN = 256
EPOCHS = 1
BATCH = 32
LR = 3e-5
SEED = 42
RUN_ID = 'arabert_text_notebook'
OUT = TASK1 / 'predictions' / 'text_1a.tsv'

for fn in (random.seed, np.random.seed, torch.manual_seed):
    fn(SEED)
torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. Load splits

In [ ]:
spec = get_task(SUBTASK)
train_records = read_jsonl(DATA / 'splits' / 'train.jsonl')
dev_records = read_jsonl(DATA / 'splits' / 'dev.jsonl')
target_records = read_jsonl(DATA / 'splits' / 'dev_test.jsonl')  # leaderboard target
print(f'train={len(train_records)}  dev={len(dev_records)}  target={len(target_records)}')
print(f'subtask={spec.name}  num_labels={spec.num_labels}  classes={spec.classes}')

train=3500  dev=500  target=500
subtask=subtask_1a  num_labels=2  classes=('Not Hateful', 'Hateful')


## 2. Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDS(Dataset):
    def __init__(self, records, spec, tokenizer, max_len):
        self.records = records; self.spec = spec; self.tok = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        enc = self.tok(r.get('text') or '', padding='max_length', truncation=True,
                       max_length=self.max_len, return_tensors='pt')
        out = {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0)}
        label = r.get('label')
        out['labels'] = -100 if label is None else self.spec.label2id[label]
        return out

train_ds = TextDS(train_records, spec, tokenizer, MAX_LEN)
dev_ds = TextDS(dev_records, spec, tokenizer, MAX_LEN)
target_ds = TextDS(target_records, spec, tokenizer, MAX_LEN)
print('built datasets')

built datasets


## 3. Model and Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=spec.num_labels,
    id2label={int(i): n for i, n in spec.id2label.items()},
    label2id=dict(spec.label2id),
    problem_type=spec.problem_type,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=str(TASK1 / 'results' / 'notebook_text'),
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=2*BATCH,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.06, weight_decay=0.01,
    logging_steps=50, save_total_limit=1, push_to_hub=False, report_to='none',
    metric_for_best_model='eval_loss', greater_is_better=False,
    seed=SEED, data_seed=SEED, remove_unused_columns=False, dataloader_num_workers=2,
)

def collate(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels': torch.tensor([b['labels'] for b in batch], dtype=torch.long),
    }

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=collate, processing_class=tokenizer,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transfo

## 4. Train

In [ ]:
trainer.train()
print('dev metrics:', trainer.evaluate())

/export/home/alishahrour/anaconda3/envs/ArGuard/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.254471,1.151851


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/export/home/alishahrour/anaconda3/envs/ArGuard/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## 5. Predict on dev_test and write a submission file

In [ ]:
p = trainer.predict(target_ds)
logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
pred_ids = logits.argmax(axis=1)
rows = [(r['id'], spec.id2label[int(pi)]) for r, pi in zip(target_records, pred_ids)]
OUT.parent.mkdir(parents=True, exist_ok=True)
write_subtask_a1_tsv(rows, OUT, run_id=RUN_ID)
print('wrote', OUT, '— first lines:')
print(OUT.read_text().splitlines()[:5])

## 6. Validate the format and (optionally) score locally

In [ ]:
import subprocess
print(subprocess.check_output(
    [sys.executable, str(TASK1 / 'format_checker' / 'format_checker.py'),
     '--subtask', SUBTASK, '--predictions', str(OUT)],
    text=True,
))

## 7. Adapting to Subtask 1B

To run this notebook for **Subtask 1B** (multi-label, unified hateful + non-hateful sub-types):

1. Set `SUBTASK = '1b'` and change `OUT` to a `.jsonl` path.
2. Filter `train_records` and `dev_records` by binary label (`spec.filter_binary`) for training.
3. Replace `CrossEntropyLoss` with `BCEWithLogitsLoss` and one-hot the targets.
4. Use `io_utils.write_multilabel_jsonl` to write predictions.

Everything is already wired in [`baselines/train_text.py`](../train_text.py) — the script is the recommended path for non-binary subtasks because it implements `pos_weight`, threshold tuning, and proper filtering.
